## Data Pre-Processing:

Using LLM to rewrite Products into a Standard format.

In [3]:
# Imports:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override= True)

True

### Choosing The Dataset:

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

- For this lab:

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [4]:
LITE_MODE = False

In [5]:
# Downloading Curated Dataset from Course Instructor from Hugging Face:
username = 'ed-donner'
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items)} items")

Loaded 820000 items


In [6]:
print(items[0])

title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", 

In [7]:
# Giving every Item a Unique ID:
for index, item in enumerate(items):
    item.id = index

In [8]:
items[0].id

0

### Defining System Prompt and Messages:

In [9]:
SYSTEM_PROMPT = """
Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features
"""

In [10]:
# Using LLM from Groq to write Standard Description of One Product:

messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': items[0].full}]

response = completion(messages= messages,
                      model= 'groq/openai/gpt-oss-20b',
                      reasoning_effort= 'low')

print(response.choices[0].message.content)
print(f"Input Tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100} Cents.")

Title: Schlage Interior Knob with Deadbolt, Oil Rubbed Bronze (Half)  
Category: Hardware  
Brand: Schlage  
Description: A half‑size interior door knob set featuring a deadbolt, crafted from oil‑rubbed bronze for a classic look.  
Details: Designed for 4” minimum center‑to‑center doors, it offers easy installation and a lifetime mechanical and finish warranty.
Input Tokens: 447
Output Tokens: 152
Cost: 0.0079125 Cents.


In [12]:
# Using Local OLLAMA Model to write Standard Description of One Product:

messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': items[0].full}]

response = completion(messages= messages,
                      model= 'ollama/llama3.1:8b',
                      api_base= 'http://172.23.37.63:11434')

print(response.choices[0].message.content)
print(f"Input Tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100} Cents.")

### System:

**Title:** Schlage Interior Deadbolt Knob
**Category:** Security and Safety
**Brand:** Schlage
**Description:** A lockset featuring a deadbolt and interior knob for added security and peace of mind.
**Details:** Precision engineered with a 100% solid metal construction.
Input Tokens: 391
Output Tokens: 63
Cost: 0.0 Cents.


### Function to Make JSON line for given Product:

In [13]:
MODEL = 'openai/gpt-oss-20b'

In [14]:
def make_jsonl(item):
    body = {'model': MODEL,
            'messages': [{'role': 'system', 'content': SYSTEM_PROMPT},
                         [{'role': 'user', 'content': item.full}]],
            'reasoning_effort': 'low'}

    line = {'custom_id': str(item.id),
            'method': 'POST',
            'url': 'v1/chat/completions',
            'body': body}

    return json.dumps(line)

In [15]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [16]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "\\nCreate a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features\\n"}, [{"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two p